In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline





In [2]:
#  DATA LOADING & PREPROCESSING

print(" Loading dataset ")
df = pd.read_csv('component_demand_history.csv')
# normalize types and sort to ensure groupby shifts are correct
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df = df.sort_values(by=['sku_id', 'week_start_date']).reset_index(drop=True)
# compute non-leaky 4-week-ahead target from historical demand_qty
df['target_4w'] = df.groupby('sku_id')['demand_qty'].transform(lambda x: x.shift(-1).rolling(window=4, min_periods=1).sum())
df.head()

Loading dataset


,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0


In [3]:
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df.head()

,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0


In [4]:
df = df.sort_values(by=['sku_id', 'week_start_date']).reset_index(drop=True)

df.head()

,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0


In [5]:
print("Imputing missing demand values")
df['demand_qty'] = df.groupby('sku_id')['demand_qty'].transform(
    lambda x: x.fillna(x.median() if not np.isnan(x.median()) else 0)
)
df.head()

Imputing missing demand values


,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0


In [6]:
# FEATURE ENGINEERING
print("Engineering features...")
df['Year'] = df['week_start_date'].dt.year
df['Month'] = df['week_start_date'].dt.month
df['Week'] = df['week_start_date'].dt.isocalendar().week.astype(int)

df.head()

Engineering features...


,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w,Year,Month,Week
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,1
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,2
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,3
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,4
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0,2024,1,5


In [7]:
df['Lag_1'] = df.groupby('sku_id')['demand_qty'].shift(1)
df['Lag_2'] = df.groupby('sku_id')['demand_qty'].shift(2)
df['Lag_3'] = df.groupby('sku_id')['demand_qty'].shift(3)

df.head()

,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w,Year,Month,Week,Lag_1,Lag_2,Lag_3
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,1,NaN,NaN,NaN
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,2,0.0,NaN,NaN
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,3,0.0,0.0,NaN
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,4,0.0,0.0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0,2024,1,5,0.0,0.0,0.0


In [8]:
df['Rolling_Mean_3'] = df.groupby('sku_id')['demand_qty'].transform(lambda x: x.shift(1).rolling(window=3).mean())
df['Rolling_Std_3'] = df.groupby('sku_id')['demand_qty'].transform(lambda x: x.shift(1).rolling(window=3).std())

df.head()

,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w,Year,Month,Week,Lag_1,Lag_2,Lag_3,Rolling_Mean_3,Rolling_Std_3
0,2024-01-01,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,1,NaN,NaN,NaN,NaN,NaN
1,2024-01-08,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,2,0.0,NaN,NaN,NaN,NaN
2,2024-01-15,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,3,0.0,0.0,NaN,NaN,NaN
3,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,4,0.0,0.0,0.0,0.0,0.0
4,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0,2024,1,5,0.0,0.0,0.0,0.0,0.0


In [9]:
df = df.dropna().reset_index(drop=True)
df.head()


,week_start_date,sku_id,product_family,demand_qty,future_4wk_avg_demand,unit_cost,on_hand_qty,supplier_id,lead_time_days,stockout_flag,target_4w,Year,Month,Week,Lag_1,Lag_2,Lag_3,Rolling_Mean_3,Rolling_Std_3
0,2024-01-22,CB-1001,Trip Unit,0.0,0.0,154.9,2,SUP-101,28,0,0.0,2024,1,4,0.0,0.0,0.0,0.0,0.0
1,2024-01-29,CB-1001,Trip Unit,0.0,0.0,154.9,24,SUP-101,28,0,0.0,2024,1,5,0.0,0.0,0.0,0.0,0.0
2,2024-02-05,CB-1001,Trip Unit,0.0,0.0,154.9,47,SUP-101,28,0,0.0,2024,2,6,0.0,0.0,0.0,0.0,0.0
3,2024-02-12,CB-1001,Trip Unit,0.0,0.0,154.9,47,SUP-101,28,0,0.0,2024,2,7,0.0,0.0,0.0,0.0,0.0
4,2024-02-19,CB-1001,Trip Unit,0.0,2.2,154.9,47,SUP-101,28,0,9.0,2024,2,8,0.0,0.0,0.0,0.0,0.0


In [10]:
#TIME-BASED VALIDATION SPLIT
split_date = df['week_start_date'].max() - pd.Timedelta(weeks=4)
train_df = df[df['week_start_date'] <= split_date]
val_df = df[df['week_start_date'] > split_date]


C:\Users\13016698\AppData\Local\Temp\ipykernel_30404\268521338.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  split_date = df['week_start_date'].max() - pd.Timedelta(weeks=4)


In [11]:
# Define feature groups
categorical_features = ['product_family', 'supplier_id']
numeric_features = [
    'Year', 'Month', 'Week', 'unit_cost', 'on_hand_qty',
    'lead_time_days', 'stockout_flag', 'Lag_1', 'Lag_2', 'Lag_3',
    'Rolling_Mean_3', 'Rolling_Std_3'
]
target = 'target_4w'

In [12]:
# Create copies for explicit conversion
X_train_raw = train_df[categorical_features + numeric_features].copy()
y_train = train_df[target]
X_val_raw = val_df[categorical_features + numeric_features].copy()
y_val = val_df[target]

In [13]:
# PREPROCESSING PIPELINE (For LR, RF, XGB)
# Linear models and tree algorithms handle categoricals differently.
# We create a preprocessor to One-Hot Encode string columns for stability.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)


In [14]:
print("Transforming features for general ML models...")
X_train_trans = preprocessor.fit_transform(X_train_raw)
X_val_trans = preprocessor.transform(X_val_raw)

Transforming features for general ML models...


In [15]:
# Native Category Conversion for LightGBM
X_train_lgb = X_train_raw.copy()
X_val_lgb = X_val_raw.copy()
for col in categorical_features:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_val_lgb[col] = X_val_lgb[col].astype('category')

In [16]:
#MULTI-MODEL TRAINING & EVALUATION
results = {}
val_predictions = {}


In [18]:
def rmse(y_true, y_pred):

    return np.sqrt(mean_squared_error(y_true, y_pred))

In [20]:
# 1. Linear Regression
print("\n Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train_trans, y_train)
preds_lr = lr.predict(X_val_trans)
results['Linear Regression'] = rmse(y_val, preds_lr)
val_predictions['Linear Regression'] = preds_lr



 Training Linear Regression...


In [21]:
# 2. Random Forest
print(" Training Random Forest (this may take a moment)...")
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_trans, y_train)
preds_rf = rf.predict(X_val_trans)
results['Random Forest'] = rmse(y_val, preds_rf)
val_predictions['Random Forest'] = preds_rf


 Training Random Forest (this may take a moment)...


In [22]:
# 3. XGBoost
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_trans, y_train)
preds_xgb = xgb_model.predict(X_val_trans)
results['XGBoost'] = rmse(y_val, preds_xgb)
val_predictions['XGBoost'] = preds_xgb


Training XGBoost...


In [23]:
# 4.LightGBM (Using fixed, non-deprecated native training structure)
print(" Training LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)])
preds_lgb = lgb_model.predict(X_val_lgb)
results['LightGBM'] = rmse(y_val, preds_lgb)
val_predictions['LightGBM'] = preds_lgb


 Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000352 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1621
[LightGBM] [Info] Number of data points in the train set: 19414, number of used features: 14
[LightGBM] [Info] Start training from score 236.906614


C:\Users\13016698\Documents\Project\.rak1venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [24]:
# 5. Ensemble Blend (Average of top performant models)
print(" Creating Ensemble Blend...")
preds_blend = (preds_xgb + preds_lgb + preds_rf) / 3
results['Ensemble Blend (RF+XGB+LGB)'] = rmse(y_val, preds_blend)


 Creating Ensemble Blend...


In [25]:

# FINAL PERFORMANCE LEADERBOARD

print("\n MODEL results ")
print("-" * 45)
for model_name, rmse_val in sorted(results.items(), key=lambda item: item[1]):
    print(f"{model_name:<30} | Validation RMSE: {rmse_val:.4f}")
print("-" * 45)



 MODEL results 
---------------------------------------------
Linear Regression              | Validation RMSE: 42.1422
Random Forest                  | Validation RMSE: 42.1434
Ensemble Blend (RF+XGB+LGB)    | Validation RMSE: 42.5960
XGBoost                        | Validation RMSE: 42.7781
LightGBM                       | Validation RMSE: 43.3031
---------------------------------------------
